# Gym-Duckietown: lane following, Pure Pursuit, and falsifying failure modes

This notebook introduces [gym-duckietown](https://github.com/duckietown/gym-duckietown): the local env, the **Pure Pursuit** lane-following baseline (imitation-learning expert), and a **simple falsification** search (seeds and steering gain) to break the claim that the default controller always stays drivable.

**Setup** (from repo root):

```bash
python3.10 -m venv .venv && source .venv/bin/activate
pip install -e .
pip install jupyter imageio imageio-ffmpeg
```

Use Jupyter with that interpreter (`gym-duckietown/.venv/bin/python`).

**Maintained fork:** see `FORK_NOTES.md` in the repo for NumPy/Pyglet/PWM/macOS fixes. Prefer **Python 3.10+**.


In [ ]:
import os
import sys

import gym
import numpy as np

REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(REPO, "learning", "imitation", "iil-dagger"))

import gym_duckietown  # registers envs; loads PWM/NumPy compatibility patch
from IPython.display import Video, display
from teacher.pure_pursuit_policy import PurePursuitPolicy


### Pure Pursuit baseline

The imitation-learning stack uses a **Pure Pursuit** controller: it tracks a lookahead point on the lane centerline (from simulator state, not from pixels). Steering is proportional to lateral error; velocity is reduced in curves. Examples scale the steering command with **`omega_gain`** (often ~7).

**Falsification:** hypothesis *H* = “with default-ish parameters, the car stays in a valid pose until `max_steps`.” Search over `(seed, omega_gain, env_id)` until `done` with an “invalid pose” message.


In [ ]:
def rollout_pp(env_id: str, seed: int, omega_gain: float, ref_v: float = 0.7, fd: float = 0.24, max_steps: int = 500):
    env = gym.make(env_id, disable_env_checker=True)
    env.seed(seed)
    obs = env.reset()
    pol = PurePursuitPolicy(env.unwrapped, ref_velocity=ref_v, following_distance=fd)
    total_r = 0.0
    last_msg = ""
    for t in range(max_steps):
        raw = pol.predict(obs)
        a = np.array([float(raw[0]), float(raw[1]) * omega_gain], dtype=np.float32)
        obs, r, done, info = env.step(a)
        total_r += r
        last_msg = info.get("Simulator", {}).get("msg", "")
        if done:
            env.close()
            return t + 1, total_r, last_msg
    env.close()
    return max_steps, total_r, last_msg


def falsify_invalid_pose(env_id, seeds=range(0, 15), omegas=(7.0, 12.0, 18.0)):
    for og in omegas:
        for s in seeds:
            steps, ret, msg = rollout_pp(env_id, s, og)
            if "invalid" in msg.lower():
                return {"seed": s, "omega_gain": og, "steps": steps, "msg": msg}
    return None

print("small_loop", falsify_invalid_pose("Duckietown-small_loop-v0"))
print("loop_obstacles", falsify_invalid_pose("Duckietown-loop_obstacles-v0", seeds=range(0, 8)))

### Two recorded failure modes

1. **High steering gain** on `Duckietown-small_loop-v0`: large `omega_gain` overshoots lane tracking and leaves the drivable mesh (`invalid pose`).
2. **`loop_obstacles` map** with default gain: Pure Pursuit follows the lane curve in pose space but does not “see” obstacles; the vehicle ends on a **floor** tile beside the road.

Run `python scripts/record_pp_failures.py` from the repo root to regenerate MP4s in `recordings/`.


In [ ]:
rec = os.path.join(REPO, "recordings")
v1 = os.path.join(rec, "failure_high_omega_gain.mp4")
v2 = os.path.join(rec, "failure_obstacles_map.mp4")
for v in (v1, v2):
    if os.path.isfile(v):
        display(Video(v, embed=True, width=640))
    else:
        print("Missing:", v)